# Training penerjemah BISINDO per kata (BiLSTM)

Jalankan di Google Colab: **Runtime > Run all**. GPU tidak wajib (datanya kecil), CPU pun selesai dalam beberapa menit.

File yang dibutuhkan (hasil `extract_features.py` di lokal):
- `data/processed/dataset.npz`
- `data/processed/labels.json`
- `features.py` (dari root repo, supaya fitur training dan aplikasi identik)

Hasil akhir: `bisindo_model.zip` berisi `model.onnx`, `labels.json`, `norm_stats.json`. Ekstrak ketiganya ke folder `assets/` di repo.

## 1. Siapkan file
Default: upload manual. Untuk memakai Google Drive, set `USE_DRIVE = True` dan taruh ketiga file di `DRIVE_DIR`.

In [ ]:
%pip install -q onnx onnxruntime

import os, shutil

USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/bisindo"
NEEDED = ["dataset.npz", "labels.json", "features.py"]

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for name in NEEDED:
        shutil.copy(os.path.join(DRIVE_DIR, name), name)
elif not all(os.path.exists(n) for n in NEEDED):
    from google.colab import files
    print("Pilih 3 file sekaligus:", ", ".join(NEEDED))
    files.upload()

missing = [n for n in NEEDED if not os.path.exists(n)]
assert not missing, f"File belum ada: {missing}"

## 2. Import dan seed

In [ ]:
import copy, json, random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import features as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

## 3. Muat data
`seqs` berisi sekuens landmark ternormalisasi dengan panjang asli (belum di-resample), supaya augmentasi waktu bisa dilakukan sebelum resample ke 32 frame. Split sudah ditentukan per video saat ekstraksi.

In [ ]:
data = np.load("dataset.npz")
labels = json.load(open("labels.json"))
y_all = data["y"].astype(np.int64)
split = data["split"]
seqs = np.split(data["seq_concat"], np.cumsum(data["seq_len"])[:-1])
assert len(seqs) == len(y_all) and seqs[0].shape[1] == F.BASE_DIM

idx = {s: np.where(split == s)[0] for s in ("train", "val", "test")}
X_clean = np.stack([F.to_model_input(s) for s in seqs])  # tanpa augmentasi
for s, ii in idx.items():
    print(f"{s:5s}: {len(ii):2d} video |", {labels[c]: int((y_all[ii] == c).sum()) for c in range(len(labels))})

## 4. Augmentasi (hanya untuk training)
- geser batas awal/akhir segmen, time-warp (kecepatan tidak seragam), frame drop
- rotasi, scale, shift kecil; noise gaussian kecil
- simulasi deteksi tangan yang sesekali gagal (lebih sering terjadi di webcam)

Tidak ada flip horizontal. Catatan: time-stretch seragam tidak berpengaruh karena semua segmen di-resample ke 32 frame, jadi yang dipakai adalah time-warp tidak seragam.

In [ ]:
HAND_SLICES = ((F.L_SL, F.FLAG_SL.start), (F.R_SL, F.FLAG_SL.start + 1))


def augment(seq, rng):
    seq = seq.copy()

    # Batas segmen: kamera tidak selalu memotong tepat di awal/akhir gerakan
    n = len(seq)
    cut = max(1, n // 10)
    a, b = rng.integers(0, cut + 1), n - rng.integers(0, cut + 1)
    if b - a >= F.MIN_SEG_FRAMES:
        seq = seq[a:b]

    # Time-warp: kecepatan lokal 0.7x-1.3x
    n = len(seq)
    speed = np.interp(np.linspace(0, 1, n - 1), np.linspace(0, 1, 4), rng.uniform(0.7, 1.3, 4))
    pos = np.concatenate([[0.0], np.cumsum(speed)])
    seq = seq[np.round(pos / pos[-1] * (n - 1)).astype(int)]

    # Frame drop sampai 20%
    keep = rng.random(len(seq)) > rng.uniform(0, 0.2)
    if keep.sum() >= F.MIN_SEG_FRAMES:
        seq = seq[keep]

    # Deteksi tangan gagal di beberapa frame
    if rng.random() < 0.5:
        for sl, flag in HAND_SLICES:
            miss = rng.random(len(seq)) < 0.05
            seq[miss, flag] = 0.0

    # Rotasi, scale, shift (di ruang ternormalisasi: 1.0 = lebar bahu)
    pts = seq[:, :F.COORD_DIM].reshape(len(seq), -1, 3)
    th = np.deg2rad(rng.uniform(-10, 10))
    rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]], np.float32)
    pts[:, :, :2] = pts[:, :, :2] @ rot.T
    pts *= rng.uniform(0.9, 1.1)
    pts[:, :, :2] += rng.normal(0, 0.05, 2).astype(np.float32)
    pts += rng.normal(0, 0.01, pts.shape).astype(np.float32)
    seq[:, :F.COORD_DIM] = pts.reshape(len(seq), -1)

    # Tangan yang tidak terdeteksi harus tetap nol
    for sl, flag in HAND_SLICES:
        seq[seq[:, flag] == 0, sl] = 0.0

    return F.to_model_input(seq)

In [ ]:
AUG_MULT = 20  # tiap video training diperbanyak 20x per epoch


class TrainSet(Dataset):
    def __init__(self, ii):
        self.ii = ii
        self.rng = np.random.default_rng(SEED)

    def __len__(self):
        return len(self.ii) * AUG_MULT

    def __getitem__(self, k):
        i = self.ii[k % len(self.ii)]
        return torch.from_numpy(augment(seqs[i], self.rng)), int(y_all[i])


g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(TrainSet(idx["train"]), batch_size=64, shuffle=True, generator=g)
print("sampel per epoch:", len(train_loader.dataset))

## 5. Model
Linear projection -> BiLSTM 2 layer (hidden 64, dropout 0.3) -> attention pooling -> FC.
Standardisasi fitur (mean/std dari data training) ditanam di dalam model, jadi aplikasi cukup mengirim fitur dari `features.py` apa adanya.

In [ ]:
class SignModel(nn.Module):
    def __init__(self, feat_dim, n_classes, mean, std, hidden=64, layers=2, dropout=0.3):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32))
        self.proj = nn.Sequential(nn.Linear(feat_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.attn = nn.Linear(2 * hidden, 1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, n_classes)

    def forward(self, x):
        x = (x - self.mean) / self.std
        h, _ = self.lstm(self.proj(x))
        w = torch.softmax(self.attn(h), dim=1)
        return self.fc(self.drop((w * h).sum(dim=1)))


flat = X_clean[idx["train"]].reshape(-1, F.FEAT_DIM)
feat_mean = flat.mean(0)
feat_std = np.maximum(flat.std(0), 1e-2)  # fitur yang selalu nol tidak meledak

model = SignModel(F.FEAT_DIM, len(labels), feat_mean, feat_std).to(device)
print("parameter:", sum(p.numel() for p in model.parameters()))

## 6. Training
Class weights untuk ketidakseimbangan kelas, early stopping berdasarkan loss validasi.

In [ ]:
MAX_EPOCHS, PATIENCE = 60, 10

counts = np.bincount(y_all[idx["train"]], minlength=len(labels))
class_w = torch.tensor(counts.sum() / (len(labels) * counts), dtype=torch.float32, device=device)
print("class weights:", dict(zip(labels, class_w.cpu().numpy().round(3))))

crit = nn.CrossEntropyLoss(weight=class_w)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)


@torch.no_grad()
def predict_proba(ii):
    model.eval()
    x = torch.from_numpy(X_clean[ii]).to(device)
    return torch.softmax(model(x), dim=-1).cpu().numpy()


@torch.no_grad()
def val_loss_acc(ii):
    model.eval()
    x = torch.from_numpy(X_clean[ii]).to(device)
    yb = torch.from_numpy(y_all[ii]).to(device)
    logits = model(x)
    return crit(logits, yb).item(), (logits.argmax(1) == yb).float().mean().item()


history, best, best_state, bad = [], float("inf"), None, 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    tl, tc, tn = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = crit(logits, yb)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item() * len(yb)
        tc += (logits.argmax(1) == yb).sum().item()
        tn += len(yb)
    vl, va = val_loss_acc(idx["val"])
    history.append((tl / tn, tc / tn, vl, va))
    print(f"epoch {epoch:2d} | train loss {tl / tn:.4f} acc {tc / tn:.3f} | val loss {vl:.4f} acc {va:.3f}")
    if vl < best - 1e-4:
        best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"early stopping di epoch {epoch} (val loss terbaik {best:.4f})")
            break

model.load_state_dict(best_state)
h = np.array(history)
plt.figure(figsize=(6, 3))
plt.plot(h[:, 0], label="train loss")
plt.plot(h[:, 2], label="val loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

## 7. Evaluasi
Confusion matrix dan akurasi per kelas pada data validasi dan test (video yang tidak pernah dilihat saat training).
Test set hanya 13 video, jadi satu video salah = turun sekitar 8%.

In [ ]:
def evaluate(name, ii):
    probs = predict_proba(ii)
    pred, true = probs.argmax(1), y_all[ii]
    cm = np.zeros((len(labels), len(labels)), int)
    for t, p in zip(true, pred):
        cm[t, p] += 1
    print(f"\n{name}: akurasi {(pred == true).mean():.1%} ({(pred == true).sum()}/{len(true)})")
    for c, lab in enumerate(labels):
        print(f"  {lab:12s} {cm[c, c]}/{cm[c].sum()} benar")
    conf = probs.max(1)
    print(f"  confidence: min {conf.min():.3f}, median {np.median(conf):.3f}")
    for i in np.where(pred != true)[0]:
        print(f"  SALAH: {data['files'][ii[i]]} -> {labels[pred[i]]} ({conf[i]:.2f})")

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("prediksi")
    ax.set_ylabel("sebenarnya")
    ax.set_title(name)
    for r in range(len(labels)):
        for c in range(len(labels)):
            ax.text(c, r, cm[r, c], ha="center", va="center",
                    color="white" if cm[r, c] > cm.max() / 2 else "black")
    plt.show()
    return probs


_ = evaluate("validasi", idx["val"])
test_probs = evaluate("test", idx["test"])

## 8. Ekspor ONNX
Output model ONNX adalah probabilitas (softmax sudah di dalam). Input: `features` berukuran (1, 32, 298).
Hasil ONNX dicek terhadap PyTorch pada data test.

In [ ]:
import onnxruntime as ort


class ExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return torch.softmax(self.m(x), dim=-1)


os.makedirs("export", exist_ok=True)
wrapper = ExportWrapper(copy.deepcopy(model).cpu()).eval()
dummy = torch.zeros(1, F.T, F.FEAT_DIM)
kw = dict(input_names=["features"], output_names=["probs"], opset_version=17)
try:
    torch.onnx.export(wrapper, (dummy,), "export/model.onnx", dynamo=False, **kw)
except TypeError:  # torch lama tanpa argumen dynamo
    torch.onnx.export(wrapper, (dummy,), "export/model.onnx", **kw)

sess = ort.InferenceSession("export/model.onnx", providers=["CPUExecutionProvider"])
onnx_probs = np.concatenate([sess.run(None, {"features": X_clean[i:i + 1]})[0] for i in idx["test"]])
diff = np.abs(onnx_probs - test_probs).max()
print(f"selisih maksimum ONNX vs PyTorch: {diff:.2e}")
assert diff < 1e-3, "hasil ONNX berbeda dari PyTorch"
assert (onnx_probs.argmax(1) == test_probs.argmax(1)).all()

json.dump(labels, open("export/labels.json", "w"), ensure_ascii=False, indent=2)
json.dump({
    "T": F.T, "feat_dim": F.FEAT_DIM,
    "catatan": "mean/std sudah ditanam di dalam model.onnx, jangan diterapkan lagi di aplikasi",
    "mean": feat_mean.tolist(), "std": feat_std.tolist(),
}, open("export/norm_stats.json", "w"))
print(os.listdir("export"), f"| model.onnx {os.path.getsize('export/model.onnx') / 1024:.0f} KB")

## 9. Unduh
Ekstrak isi zip ke folder `assets/` di repo.

In [ ]:
shutil.make_archive("bisindo_model", "zip", "export")
if USE_DRIVE:
    shutil.copy("bisindo_model.zip", DRIVE_DIR)
    print("tersimpan di", DRIVE_DIR)
try:
    from google.colab import files
    files.download("bisindo_model.zip")
except ImportError:
    print("bukan di Colab: ambil bisindo_model.zip secara manual")